In [1]:
import librosa
import librosa.display
import numpy as np
import pandas as pd
import os
from typing import List, Tuple, Dict
import matplotlib.pyplot as plt
import random
import cv2
# from vgg16 import VGG16_LSTM
import pickle
import torch.nn as nn
import torch
import datetime
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms, utils, models
from torch.utils.data import Dataset, DataLoader
from facenet_pytorch import MTCNN
import logging
import torch
import torch.nn as nn
from vivit import ViViTBackbone
import gc
from einops import rearrange
from tqdm import tqdm
# from kw_r2plus1d_video_swin_transformer import SwinTransformer3D
batchsz = 4
frame_nums=15
start_epoch = 0
test_flage=False
model_name ="fullshot"
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
formatter = logging.Formatter('[%(asctime)s]::%(module)s::%(levelname)s::%(message)s')
streamHandler = logging.StreamHandler()
streamHandler.setFormatter(formatter)
fileHandler = logging.FileHandler('./LOG/personalityLog.log')
fileHandler.setFormatter(formatter)
logger.addHandler(streamHandler)
logger.addHandler(fileHandler)
USE_CUDA = torch.cuda.is_available()
# device = torch.device('cuda:1' if USE_CUDA else 'cpu')

In [2]:
logger.debug('============={}+Video Vision Transformer Strat!============='.format(model_name))
USE_CUDA = torch.cuda.is_available()
print(USE_CUDA)

device = torch.device('cuda:0' if USE_CUDA else 'cpu')
print('학습을 진행하는 기기:', device)

[2026-09-22 12:00:46,176]::3184030073::DEBUG::=============fullshot+Video Vision Transformer Strat!=============


True
학습을 진행하는 기기: cuda:0


In [3]:
def getdata(pathname,datatype):
    if datatype=="train":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            # train_data_set.extend(pickle.load(output_file))
            final_data_set.extend(pickle.load(output_file))
            # print(i)
        return final_data_set
    elif datatype=="valid":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            valid_data_set=[]
            final_data_set.extend(pickle.load(output_file))
            del valid_data_set
        return final_data_set
print(1)
def reshape_to_expected_input(dataset: List[Tuple[ np.ndarray, np.ndarray]]) -> Tuple[
     np.ndarray, np.ndarray]:
    x0_list = []
    x1_list = []
    for i in range(0, len(dataset)):
        x0_list.append(dataset[i][0])
        x1_list.append(dataset[i][1])
    return (np.stack(x0_list), np.stack(x1_list))

1


In [4]:

save_model_file_path = './save_swintransformer_folder/{}_{}.{}'
mtcnn= MTCNN(image_size=224, margin=0, min_face_size= 20, thresholds=[0.6, 0.7, 0.7], post_process=True)
print(1)

1


In [5]:
train_set_data=getdata("/data1/qx/mean_based_arg/udiva/multimodal_fusion/data/speaking_only/train_set.dat","train")
print(1)
validation_set_data=getdata("/data1/qx/mean_based_arg/udiva/multimodal_fusion/data/speaking_only/valid_set.dat","valid")
print(1)

1
1


In [6]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
])
print(1)

1


In [7]:
class ChalearnDataset(Dataset):
    def __init__(self,fullshot,tagdata,transform=None):
        self.fullshot=fullshot
        self.tagdata=tagdata
        self.transform = transform  # 표준화 여부
    def __len__(self):
        return len(self.fullshot)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()#텐서의 경우 목록으로 돌아가기
        fullshot=self.fullshot[idx]
        fullshot=torch.FloatTensor(fullshot)
        big_five_sorces=self.tagdata[idx]
        big_five_sorces = torch.FloatTensor(big_five_sorces)
        return fullshot,big_five_sorces
print(1)

1


In [8]:
num_workerssz = 20
lr = 3e-05
epochs = 120

# Multi-seed setting
SEEDS = [42, 123, 777, 1004, 2024]
patience = 20

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

print(1)


1


In [9]:
train_set_data = ChalearnDataset(
    fullshot=train_set_data[1],
    tagdata=train_set_data[2],
    transform=transform
)

val_set_data = ChalearnDataset(
    fullshot=validation_set_data[1],
    tagdata=validation_set_data[2],
    transform=transform
)

print(1)


1


In [10]:
def extract_ocean(data,batch_size):
    o=[]
    c=[]
    e=[]
    a=[]
    n=[]
    if batch_size >1:
        for i in range(batch_size):
            o.append(data[i][0].item())
            c.append(data[i][1].item())
            e.append(data[i][2].item())
            a.append(data[i][3].item())
            n.append(data[i][4].item())
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n) 
    elif batch_size == 1:
        data=data.squeeze()
        for i in range(batch_size):
            o.append(data[0].item())
            c.append(data[1].item())
            e.append(data[2].item())
            a.append(data[3].item())
            n.append(data[4].item())
            
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n)
    return o,c,e,a,n

In [11]:
# ============================================================
# Multi-seed Training
# ============================================================

# Only ONE checkpoint file is kept.
# Whenever a better validation MSE is found across all seeds,
# this same file is overwritten.
best_model_path = "./save_swintransformer_folder/model_best_multiseed.pth"

seed_results = []
global_best_mse = float("inf")
global_best_seed = None
global_best_epoch = None

with torch.cuda.device(0):

    for seed in SEEDS:

        print("\n" + "=" * 80)
        print(f"Start training | seed = {seed}")
        print("=" * 80)

        # ----------------------------------------------------
        # Reproducibility
        # ----------------------------------------------------
        set_seed(seed)

        # Re-create DataLoaders for each seed so that the
        # shuffled training order is also controlled by seed.
        generator = torch.Generator()
        generator.manual_seed(seed)

        train_dataloader = DataLoader(
            dataset=train_set_data,
            batch_size=batchsz,
            shuffle=True,
            num_workers=num_workerssz,
            worker_init_fn=seed_worker,
            generator=generator
        )

        val_dataloader = DataLoader(
            dataset=val_set_data,
            batch_size=batchsz,
            shuffle=False,
            num_workers=num_workerssz,
            worker_init_fn=seed_worker,
            generator=generator
        )

        # ----------------------------------------------------
        # IMPORTANT:
        # Every seed starts from a newly initialized model.
        # ----------------------------------------------------
        model = ViViTBackbone(
            2, 14, 14, 2, 2, 2, 5, 96, 6, 10, 8, model=3
        ).to(device)

        criterion = torch.nn.MSELoss().to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

        seed_best_mse = float("inf")
        seed_best_epoch = -1
        no_improve_count = 0

        for i in range(start_epoch, epochs):

            train_avg_loss = 0
            val_avg_loss = 0

            train_avg_o_loss = 0
            train_avg_c_loss = 0
            train_avg_e_loss = 0
            train_avg_a_loss = 0
            train_avg_n_loss = 0

            val_avg_o_loss = 0
            val_avg_c_loss = 0
            val_avg_e_loss = 0
            val_avg_a_loss = 0
            val_avg_n_loss = 0

            # ------------------------------------------------
            # Train
            # ------------------------------------------------
            model.train()

            for fullshot, big_five_data in tqdm(
                train_dataloader,
                desc=f"Seed {seed} | Train {i + 1}/{epochs}"
            ):
                fullshot = fullshot.permute(0, 4, 1, 2, 3)
                big_five_data = big_five_data.permute(0, 2, 1)
                big_five_data = big_five_data.squeeze()

                fullshot = fullshot.to(device)
                big_five_data = big_five_data.to(device)

                optimizer.zero_grad()

                hypothesis = model(fullshot)
                loss = criterion(hypothesis, big_five_data)

                loss.backward()
                optimizer.step()

                train_avg_loss += loss

            train_avg_loss = train_avg_loss / len(train_dataloader)

            # ------------------------------------------------
            # Validation
            # ------------------------------------------------
            model.eval()

            with torch.no_grad():

                for fullshot, big_five_data in tqdm(
                    val_dataloader,
                    desc=f"Seed {seed} | Valid {i + 1}/{epochs}"
                ):
                    fullshot = fullshot.permute(0, 4, 1, 2, 3)
                    big_five_data = big_five_data.permute(0, 2, 1)
                    big_five_data = big_five_data.squeeze()

                    fullshot = fullshot.to(device)
                    big_five_data = big_five_data.to(device)

                    hypothesis = model(fullshot)
                    val_loss = criterion(hypothesis, big_five_data)

                    hypothesiso, hypothesisc, hypothesise, hypothesisa, hypothesisn = \
                        extract_ocean(hypothesis, len(fullshot))

                    big_five_datao, big_five_datac, big_five_datae, \
                        big_five_dataa, big_five_datan = \
                        extract_ocean(big_five_data, len(fullshot))

                    val_o_loss = criterion(hypothesiso, big_five_datao)
                    val_c_loss = criterion(hypothesisc, big_five_datac)
                    val_e_loss = criterion(hypothesise, big_five_datae)
                    val_a_loss = criterion(hypothesisa, big_five_dataa)
                    val_n_loss = criterion(hypothesisn, big_five_datan)

                    val_avg_loss += val_loss
                    val_avg_o_loss += val_o_loss
                    val_avg_c_loss += val_c_loss
                    val_avg_e_loss += val_e_loss
                    val_avg_a_loss += val_a_loss
                    val_avg_n_loss += val_n_loss

                val_avg_loss = val_avg_loss / len(val_dataloader)
                val_avg_o_loss = val_avg_o_loss / len(val_dataloader)
                val_avg_c_loss = val_avg_c_loss / len(val_dataloader)
                val_avg_e_loss = val_avg_e_loss / len(val_dataloader)
                val_avg_a_loss = val_avg_a_loss / len(val_dataloader)
                val_avg_n_loss = val_avg_n_loss / len(val_dataloader)

            # Convert to Python float for comparisons / logging.
            current_val_mse = val_avg_loss.item()

            # ------------------------------------------------
            # Best result of the CURRENT seed (lower MSE is better)
            # ------------------------------------------------
            if current_val_mse < seed_best_mse:
                seed_best_mse = current_val_mse
                seed_best_epoch = i + 1
                no_improve_count = 0
            else:
                no_improve_count += 1

            # ------------------------------------------------
            # Global best checkpoint across ALL seeds
            # ------------------------------------------------
            if current_val_mse < global_best_mse:
                global_best_mse = current_val_mse
                global_best_seed = seed
                global_best_epoch = i + 1

                torch.save({
                    "seed": seed,
                    "epoch": i + 1,
                    "model": model.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "loss": current_val_mse,
                }, best_model_path)

                logger.info(
                    "Global best updated | seed: {} | epoch: {} | MSE: {:.6f} |best epoch: {} |patience: {}".format(
                        seed, i + 1, current_val_mse, global_best_epoch, patience
                    )
                )

            logger.debug(
                "Seed: {} | Epoch: {} | MSE: {:.6f} | best mae: {} |patience: {} |  "
                "O-MSE: {:.6f} | C-MSE: {:.6f} | E-MSE: {:.6f} | "
                "A-MSE: {:.6f} | N-MSE: {:.6f}".format(
                    seed, i + 1, current_val_mse,global_best_mse, patience,
                    val_avg_o_loss.item(),
                    val_avg_c_loss.item(),
                    val_avg_e_loss.item(),
                    val_avg_a_loss.item(),
                    val_avg_n_loss.item()
                )
            )

            torch.cuda.empty_cache()

            # ------------------------------------------------
            # Early stopping based on validation MSE
            # ------------------------------------------------
            if no_improve_count >= patience:
                logger.info(
                    "Early stopping | seed: {} | epoch: {} | "
                    "best epoch: {} | best MSE: {:.6f}".format(
                        seed, i + 1, seed_best_epoch, seed_best_mse
                    )
                )
                break

        # ----------------------------------------------------
        # Store scalar results for this seed
        # ----------------------------------------------------
        seed_results.append({
            "seed": seed,
            "best_epoch": seed_best_epoch,
            "best_mse": seed_best_mse
        })

        print(
            f"Seed {seed} finished | "
            f"Best epoch = {seed_best_epoch} | "
            f"Best MSE = {seed_best_mse:.6f}"
            f"patience = {patience}"
        )

        # Release memory before next seed
        del model
        del optimizer
        del criterion
        del train_dataloader
        del val_dataloader
        del generator
        gc.collect()
        torch.cuda.empty_cache()


# ============================================================
# Multi-seed Summary — MSE
# ============================================================
all_mse = np.array(
    [result["best_mse"] for result in seed_results],
    dtype=np.float64
)

mean_mse = all_mse.mean()
std_mse = all_mse.std(ddof=1) if len(all_mse) > 1 else 0.0

print("\n" + "=" * 80)
print("MULTI-SEED RESULTS")
print("=" * 80)

for result in seed_results:
    print(
        "Seed {:>4} | Best Epoch {:>3} | MSE {:.6f}".format(
            result["seed"],
            result["best_epoch"],
            result["best_mse"]
        )
    )

print("-" * 80)
print(f"Mean MSE : {mean_mse:.6f}")
print(f"Std  MSE : {std_mse:.6f}")
print(
    f"Global best: seed={global_best_seed}, "
    f"epoch={global_best_epoch}, "
    f"MSE={global_best_mse:.6f}"
)
print(f"Saved checkpoint: {best_model_path}")
print("=" * 80)



Start training | seed = 42


Seed 42 | Train 1/120: 100%|██████████| 1091/1091 [04:08<00:00,  4.39it/s]
Seed 42 | Valid 1/120: 100%|██████████| 191/191 [00:16<00:00, 11.49it/s]
[2026-09-22 12:05:17,930]::4208285795::INFO::Global best updated | seed: 42 | epoch: 1 | MSE: 1.483774 |best epoch: 1 |patience: 20
[2026-09-22 12:05:17,932]::4208285795::DEBUG::Seed: 42 | Epoch: 1 | MSE: 1.483774 | best mae: 1.4837735891342163 |patience: 20 |  O-MSE: 1.013169 | C-MSE: 1.820572 | E-MSE: 1.232287 | A-MSE: 1.779656 | N-MSE: 1.573185
Seed 42 | Train 2/120: 100%|██████████| 1091/1091 [04:09<00:00,  4.37it/s]
Seed 42 | Valid 2/120: 100%|██████████| 191/191 [00:18<00:00, 10.58it/s]
[2026-09-22 12:09:46,966]::4208285795::INFO::Global best updated | seed: 42 | epoch: 2 | MSE: 1.463534 |best epoch: 2 |patience: 20
[2026-09-22 12:09:46,968]::4208285795::DEBUG::Seed: 42 | Epoch: 2 | MSE: 1.463534 | best mae: 1.463533639907837 |patience: 20 |  O-MSE: 1.103671 | C-MSE: 1.487447 | E-MSE: 1.569545 | A-MSE: 1.731991 | N-MSE: 1.425013
Seed 

Seed 42 finished | Best epoch = 10 | Best MSE = 1.290887patience = 20

Start training | seed = 123


Seed 123 | Train 1/120: 100%|██████████| 1091/1091 [04:10<00:00,  4.35it/s]
Seed 123 | Valid 1/120: 100%|██████████| 191/191 [00:18<00:00, 10.58it/s]
[2026-09-22 14:19:49,831]::4208285795::DEBUG::Seed: 123 | Epoch: 1 | MSE: 1.505171 | best mae: 1.290886640548706 |patience: 20 |  O-MSE: 1.229452 | C-MSE: 1.500582 | E-MSE: 1.372654 | A-MSE: 1.800665 | N-MSE: 1.622501
Seed 123 | Train 2/120: 100%|██████████| 1091/1091 [04:10<00:00,  4.35it/s]
Seed 123 | Valid 2/120: 100%|██████████| 191/191 [00:18<00:00, 10.54it/s]
[2026-09-22 14:24:19,474]::4208285795::DEBUG::Seed: 123 | Epoch: 2 | MSE: 1.430228 | best mae: 1.290886640548706 |patience: 20 |  O-MSE: 1.350948 | C-MSE: 1.349263 | E-MSE: 1.340663 | A-MSE: 1.682055 | N-MSE: 1.428214
Seed 123 | Train 3/120: 100%|██████████| 1091/1091 [04:10<00:00,  4.35it/s]
Seed 123 | Valid 3/120: 100%|██████████| 191/191 [00:18<00:00, 10.30it/s]
[2026-09-22 14:28:49,636]::4208285795::DEBUG::Seed: 123 | Epoch: 3 | MSE: 1.454735 | best mae: 1.290886640548706 |

Seed 123 finished | Best epoch = 13 | Best MSE = 1.310192patience = 20

Start training | seed = 777


...

Seed 777 finished | Best epoch = 25 | Best MSE = 1.266005patience = 20

Start training | seed = 1004


Seed 1004 | Train 1/120: 100%|██████████| 1091/1091 [04:11<00:00,  4.34it/s]
Seed 1004 | Valid 1/120: 100%|██████████| 191/191 [00:18<00:00, 10.15it/s]
[2026-09-22 20:11:34,268]::4208285795::DEBUG::Seed: 1004 | Epoch: 1 | MSE: 1.460963 | best mae: 1.2660053968429565 |patience: 20 |  O-MSE: 1.104418 | C-MSE: 1.487708 | E-MSE: 1.138799 | A-MSE: 1.614256 | N-MSE: 1.959632
Seed 1004 | Train 2/120: 100%|██████████| 1091/1091 [04:11<00:00,  4.34it/s]
Seed 1004 | Valid 2/120: 100%|██████████| 191/191 [00:18<00:00, 10.43it/s]
[2026-09-22 20:16:04,841]::4208285795::DEBUG::Seed: 1004 | Epoch: 2 | MSE: 1.359438 | best mae: 1.2660053968429565 |patience: 20 |  O-MSE: 1.149835 | C-MSE: 1.240633 | E-MSE: 1.215583 | A-MSE: 1.684789 | N-MSE: 1.506347
Seed 1004 | Train 3/120: 100%|██████████| 1091/1091 [04:11<00:00,  4.33it/s]
Seed 1004 | Valid 3/120: 100%|██████████| 191/191 [00:18<00:00, 10.39it/s]
[2026-09-22 20:20:35,645]::4208285795::DEBUG::Seed: 1004 | Epoch: 3 | MSE: 1.442902 | best mae: 1.266005

Seed 1004 finished | Best epoch = 26 | Best MSE = 1.278430patience = 20

Start training | seed = 2024


Seed 2024 | Train 1/120: 100%|██████████| 1091/1091 [04:09<00:00,  4.38it/s]
Seed 2024 | Valid 1/120: 100%|██████████| 191/191 [00:17<00:00, 10.63it/s]
[2026-09-22 23:37:55,545]::4208285795::DEBUG::Seed: 2024 | Epoch: 1 | MSE: 1.391319 | best mae: 1.2660053968429565 |patience: 20 |  O-MSE: 1.316477 | C-MSE: 1.278686 | E-MSE: 1.306177 | A-MSE: 1.604416 | N-MSE: 1.450837
Seed 2024 | Train 2/120: 100%|██████████| 1091/1091 [04:09<00:00,  4.37it/s]
Seed 2024 | Valid 2/120: 100%|██████████| 191/191 [00:18<00:00, 10.61it/s]
[2026-09-22 23:42:23,747]::4208285795::DEBUG::Seed: 2024 | Epoch: 2 | MSE: 1.472317 | best mae: 1.2660053968429565 |patience: 20 |  O-MSE: 1.240665 | C-MSE: 1.544850 | E-MSE: 1.504412 | A-MSE: 1.540829 | N-MSE: 1.530827
Seed 2024 | Train 3/120: 100%|██████████| 1091/1091 [04:09<00:00,  4.37it/s]
Seed 2024 | Valid 3/120: 100%|██████████| 191/191 [00:18<00:00, 10.58it/s]
[2026-09-22 23:46:52,141]::4208285795::DEBUG::Seed: 2024 | Epoch: 3 | MSE: 1.366533 | best mae: 1.266005

Seed 2024 finished | Best epoch = 7 | Best MSE = 1.270073patience = 20

MULTI-SEED RESULTS
Seed   42 | Best Epoch  10 | MSE 1.290887
Seed  123 | Best Epoch  13 | MSE 1.310192
Seed  777 | Best Epoch  25 | MSE 1.266005
Seed 1004 | Best Epoch  26 | MSE 1.278430
Seed 2024 | Best Epoch   7 | MSE 1.270073
--------------------------------------------------------------------------------
Mean MSE : 1.283117
Std  MSE : 0.017877
Global best: seed=777, epoch=25, MSE=1.266005
Saved checkpoint: ./save_swintransformer_folder/model_best_multiseed.pth
